### Setup

In [26]:
from astroquery.vizier import Vizier
import pandas as pd
import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord

from dustmaps.bayestar import BayestarQuery
from dustmaps.edenhofer2023 import Edenhofer2023Query
from dustmaps.config import config

Vizier.ROW_LIMIT = -1


In [ ]:
# Replace with your own local paths and ID column names as needed

hosts_folder = '/Users/philvanlane/Documents/ax_M/'
hosts = pd.read_csv(hosts_folder + 'test_dereddenings.csv', dtype={'Source': str})
gids = hosts['object'].tolist()

## De-reddening

### Step 1. Get distances from Bailer-Jones catalog based on Gaia ID

This requires querying the Vizier catalog. This is sample code to generate a DataFrame with the Bailer-Jones fields for all Gaia IDs that can be found in that catalogue.

In [7]:
result = Vizier.query_constraints(
        catalog="I/352/gedr3dis",
        Source=gids[i]
    )

In [12]:
df = pd.DataFrame(columns=['Source', 'RA_ICRS', 'DE_ICRS', 'rgeo', 'b_rgeo', 'B_rgeo', 'rpgeo',
       'b_rpgeo', 'B_rpgeo', 'Flag'])

In [13]:
for i in range(len(gids)):
    if i % 500 == 0:
        print(f"Processing {i} of {len(gids)}")
    if len(df.loc[df['Source'] == gids[i]]) > 0:
        continue
    result = Vizier.query_constraints(
        catalog="I/352/gedr3dis",
        Source=gids[i]
    )
    if result.keys() == []:
        print(f"No result for {gids[i]}")
        continue
    df_record = result['I/352/gedr3dis'].to_pandas()
    df = pd.concat([df, df_record], ignore_index=True)

Processing 0 of 4


/var/folders/lf/sq3cxxf17kv47p5lcv6w5t5r0000gn/T/ipykernel_56501/440846398.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_record], ignore_index=True)


Now we need to convert the upper and lower 1 sigma distances to an error. In this case we use `rpgeo` as the distance, which we should do when it is available. When `rpgeo` has not been calculated we can use `rgeo`.

In [16]:
df['rgeo_err'] = (df['B_rgeo'] - df['b_rgeo']) / 2
df['rpgeo_err'] = (df['B_rpgeo'] - df['b_rpgeo']) / 2

In [18]:
df['dist_pc'] = np.where(
    np.isfinite(df['rpgeo']),
    df['rpgeo'],
    df['rgeo']
)

In [19]:
df['dist_pc_err'] = np.where(
    np.isfinite(df['rpgeo_err']),
    df['rpgeo_err'],
    df['rgeo_err']
)

### Step 2. Load dustmaps and calculate extinction values

First we need to load the dustmaps and set them up.

In [24]:
# Change this to your local dustmaps data directory
config['data_dir'] = '/Users/philvanlane/opt/anaconda3/lib/python3.9/site-packages/dustmaps/data'

# Load edenhofer map (change seed if you want to, this is used for random sampling)
edenhofer = Edenhofer2023Query(load_samples=True,integrated=False,seed=19)

# Save the Edehofer grid
ed = edenhofer.distances / (1*u.pc)
ed[0] = ed[0] + 0.0001
ed[-1] = ed[-1] - 0.0001

# Load the Bayestar dustmap
bayestar = BayestarQuery(max_samples=10)

# Set number of samples to draw based on distance and distance error
num_dist_samples = 10

Optimizing map for querying (this might take a couple of seconds)...


Loading pixel_info ...
Loading samples ...
Loading best_fit ...
Replacing NaNs in reliable distance estimates ...
Sorting pixel_info ...
Extracting hp_idx_sorted and data_idx at each nside ...
  nside = 64
  nside = 128
  nside = 256
  nside = 512
  nside = 1024
t = 34.067 s
  pix_info:   0.498 s
   samples:  16.379 s
      best:   3.256 s
       nan:   0.052 s
      sort:  13.802 s
       idx:   0.080 s


Now we actually query the dustmaps to get extinction samples for each star.

In [27]:
samples = {}

for i,r in df.iterrows():
    
    gid = str(r.Source)
    dist = r.dist_pc
    dist_err = r.dist_pc_err
    dist_samples = np.random.normal(dist, dist_err, num_dist_samples)
    
    # E samples
    
    b_all_samples = []
    e_all_samples = []
    
    
    # Getting integrated Edenhofer values
    
    max_dist = np.max(dist_samples)

    if max_dist > 0:
        ed_sub = [dist for dist in ed if dist <= max_dist]
        ed_sub.append(max_dist)

        eden_E = []

        for i2 in range(len(ed_sub)):
            coords = SkyCoord(r.RA_ICRS*u.deg,r.DE_ICRS*u.deg,ed_sub[i2]*u.pc,frame='icrs')
            eden_E.append(edenhofer(coords,mode='samples'))

        eden_E = np.array(eden_E)
    
    # Getting E for each distance
    
    for i in range(len(dist_samples)):
        d = dist_samples[i]
        if d>= 0:
            
            # Bayestar
            
            coord = SkyCoord(r.RA_ICRS*u.deg, r.DE_ICRS*u.deg, distance=d*u.pc, frame='icrs')
            E = bayestar(coord,mode='samples',return_flags=True)
            qa_flags = E[1]
            if (qa_flags[0] & qa_flags[1]):
                for e in E[0]:
                    b_all_samples.append(e)
                
                
            # Edenhofer

            index = np.searchsorted(ed_sub, dist_samples[i])

            # Slice x and y arrays up to this index
            if ed_sub[index] == dist_samples[i]:
                x_sub = ed_sub[:index+1]
                y_sub = eden_E[:index]

            # If x_limit is not exactly in the array, we interpolate the last point
            else:
                x_sub = ed_sub[:index]
                y_sub = eden_E[:index]
                x_sub = np.append(x_sub, dist_samples[i])

            for q in range(12):
                y_sub_ind_sample = y_sub[:,q]
                y_sub_ind_sample = np.append(y_sub_ind_sample,
                                             np.interp(dist_samples[i], ed_sub, eden_E[:,q]))

                # Perform the integration
                result = np.trapz(y_sub_ind_sample, x_sub)
                e_all_samples.append(np.trapz(y=y_sub_ind_sample,x=x_sub))

    b_all_samples = np.array(b_all_samples)
    e_all_samples = np.array(e_all_samples)
    
    # Add to dict
    
    samples[gid] = {
        'ra': r.RA_ICRS,
        'dec': r.DE_ICRS,
        'dist_samples': dist_samples,
        'E_samples_Bayestar': b_all_samples,
        'E_samples_Edenhofer': e_all_samples
    }

Now we calculate summary stats for each star.

In [29]:
for gid in samples.keys():
    samples[gid]['E_Bayestar_med'] = np.nanmedian(samples[gid]['E_samples_Bayestar'])
    samples[gid]['E_Bayestar_std'] = np.nanstd(samples[gid]['E_samples_Bayestar'])
    samples[gid]['E_Edenhofer_med'] = np.nanmedian(samples[gid]['E_samples_Edenhofer'])
    samples[gid]['E_Edenhofer_std'] = np.nanstd(samples[gid]['E_samples_Edenhofer'])

/Users/philvanlane/opt/anaconda3/envs/default-py312/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/philvanlane/opt/anaconda3/envs/default-py312/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Converting extinction in E units from dust maps to E(B-V):

$$E(B-V) = 0.829 \times E_{Edenhofer2023}$$

$$E(B-V) = 0.88 \times E_{Bayestar2019}$$

We adopt a $R_V$ value of 3.1 in the following:

$$A_V = R_V \times E(B-V)$$

Gaia band conversion constants:

$$A_G = (0.789 \pm 0.005) \times A_V$$

$$A_{G_{BP}} = (1.002 \pm 0.007) \times A_V$$

$$A_{G_{RP}} = (0.589 \pm 0.004) \times A_V$$


I then re-calculate the Gaia fluxes for each band _b_ by applying:

$$F_{de-red,b} = F_{obs} \times 10^{0.4A_b}$$

I do this instead of simply adding/subtracting the magnitudes because uncertainties on Gaia fluxes are provided, however uncertainties on the magnitude values are not, and flux is the more direct observational measurement.

I then convert these de-reddened fluxed back to de-reddened magnitudes by applying:

$$G_{de-red,b} = -2.5 \times \log(\frac{F_{de-red,b}}{F_{zpt,b}})$$

where $F_{zpt,b}$ is the Gaia zeropoint flux for each band.

We can then calculated de-reddened colours using:

$$(G_{BP} - G_{RP})_0 = G_{de-red,BP} - G_{de-red,RP}$$

Uncertainties are propagated throughout these calculations where available. The raw sources of uncertainty are:

$\sigma_E$: The standard deviation of sample E values from the dustmap.

$\sigma_{A_b}$: The uncertainties on the $A_V$ to $A_b$ conversion constants as supplied in literature.

$\sigma_{F_{obs,b}}$: The reported uncertainties on Gaia fluxes for each band.



#### Calculating $E_{B-V}$

In [33]:
for gid in samples.keys():
    samples[gid]['EBV_Edenhofer'] = samples[gid]['E_Edenhofer_med'] * 0.829
    samples[gid]['EBV_Edenhofer_std'] = samples[gid]['E_Edenhofer_std'] * 0.829
    samples[gid]['EBV_Bayestar'] = samples[gid]['E_Bayestar_med'] * 0.88
    samples[gid]['EBV_Bayestar_std'] = samples[gid]['E_Bayestar_std'] * 0.88

#### Calculating $A_v$

In [34]:
# Conversion constants from table 3 of Wang & Chen (and standard values)

Rv = 3.1

A_GBP_div_Av = 1.002
A_GBP_div_Av_err = 0.007
A_GBP_div_Av_frac_err = A_GBP_div_Av_err / A_GBP_div_Av

A_GRP_div_Av = 0.589
A_GRP_div_Av_err = 0.004
A_GRP_div_Av_frac_err = A_GRP_div_Av_err / A_GRP_div_Av

A_G_div_Av = 0.789
A_G_div_Av_err = 0.005
A_G_div_Av_frac_err = A_G_div_Av_err / A_G_div_Av

In [35]:
def getExtinction(EBV,EBV_err,Rv=Rv):
    
    Av = Rv * EBV
    Av_err = Rv * EBV_err

    if Av > 0:
        a = Av_err / Av
    else:
        a = 0
        
    Av_frac_err = a

    A_GRP = Av * A_GRP_div_Av
    A_GRP_frac_error = np.sqrt(Av_frac_err**2 + A_GRP_div_Av_frac_err**2)
    A_GRP_err = A_GRP_frac_error * A_GRP
    
    A_GBP = Av * A_GBP_div_Av
    A_GBP_frac_error = np.sqrt(Av_frac_err**2 + A_GBP_div_Av_frac_err**2)
    A_GBP_err = A_GBP_frac_error * A_GBP
    
    A_G = Av * A_G_div_Av
    A_G_frac_err = np.sqrt(Av_frac_err**2 + A_G_div_Av_frac_err**2)
    A_G_err = A_G_frac_err * A_G
    
    return [
            [Av,Av_err],
            [A_GBP,A_GBP_err],
            [A_GRP,A_GRP_err],
            [A_G,A_G_err]
           ]  

In [36]:
for gid in samples.keys():
    
    # Edenhofer
    [[Av,Av_err],[A_GBP,A_GBP_err],[A_GRP,A_GRP_err],[A_G,A_G_err]] = getExtinction(
        samples[gid]['EBV_Edenhofer'],
        samples[gid]['EBV_Edenhofer_std'])
    
    samples[gid]['Av_Edenhofer'] = Av
    samples[gid]['Av_Edenhofer_std'] = Av_err
    samples[gid]['AG_Edenhofer'] = A_G
    samples[gid]['AG_Edenhofer_std'] = A_G_err
    samples[gid]['ABP_Edenhofer'] = A_GBP
    samples[gid]['ABP_Edenhofer_std'] = A_GBP_err
    samples[gid]['ARP_Edenhofer'] = A_GRP
    samples[gid]['ARP_Edenhofer_std'] = A_GRP_err

    # Bayestar
    [[Av,Av_err],[A_GBP,A_GBP_err],[A_GRP,A_GRP_err],[A_G,A_G_err]] = getExtinction(
    samples[gid]['EBV_Bayestar'],
    samples[gid]['EBV_Bayestar_std'])
    
    samples[gid]['Av_Bayestar'] = Av
    samples[gid]['Av_Bayestar_std'] = Av_err
    samples[gid]['AG_Bayestar'] = A_G
    samples[gid]['AG_Bayestar_std'] = A_G_err
    samples[gid]['ABP_Bayestar'] = A_GBP
    samples[gid]['ABP_Bayestar_std'] = A_GBP_err
    samples[gid]['ARP_Bayestar'] = A_GRP
    samples[gid]['ARP_Bayestar_std'] = A_GRP_err

#### Create DataFrame from final extinction values

In [37]:
ext_df = pd.DataFrame({
    'dr3_id': samples.keys(),
    # 'dist_pc': [gid['dist_avg'] for gid in samples_new.values()],
    
    'Av_b': [gid['Av_Bayestar'] for gid in samples.values()],
    'Av_err_b': [gid['Av_Bayestar_std'] for gid in samples.values()],
    'AG_b': [gid['AG_Bayestar'] for gid in samples.values()],
    'AG_err_b': [gid['AG_Bayestar_std'] for gid in samples.values()],
    'ABP_b': [gid['ABP_Bayestar'] for gid in samples.values()],
    'ABP_err_b': [gid['ABP_Bayestar_std'] for gid in samples.values()],
    'ARP_b': [gid['ARP_Bayestar'] for gid in samples.values()],
    'ARP_err_b': [gid['ARP_Bayestar_std'] for gid in samples.values()],
    
    'Av_e': [gid['Av_Edenhofer'] for gid in samples.values()],
    'Av_err_e': [gid['Av_Edenhofer_std'] for gid in samples.values()],
    'AG_e': [gid['AG_Edenhofer'] for gid in samples.values()],
    'AG_err_e': [gid['AG_Edenhofer_std'] for gid in samples.values()],
    'ABP_e': [gid['ABP_Edenhofer'] for gid in samples.values()],
    'ABP_err_e': [gid['ABP_Edenhofer_std'] for gid in samples.values()],
    'ARP_e': [gid['ARP_Edenhofer'] for gid in samples.values()],
    'ARP_err_e': [gid['ARP_Edenhofer_std'] for gid in samples.values()],
    })

In [39]:
ext_df.to_csv(hosts_folder + 'extinction.csv', index=False)